# Free Kaggle cloud pipeline

All models run on the cloud GPU. No payment is required. See docs/cloud.md in the repository.

Enable Internet and GPU T4 x2 in Settings; complete phone verification if requested. Use Save Version > Save & Run All (Commit) for an unattended run. Check the session limit shown in your account and lower the 11-hour budget if necessary. Outputs are in /kaggle/working.


In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

repo_dir = Path("/kaggle/working/optimal-stopping-in-autoregressive-thought-chain")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/mengtobby/optimal-stopping-in-autoregressive-thought-chain.git", str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[model]"], check=True)


In [ ]:
workdir = Path("/kaggle/working/osc-runs/math500")
# To resume a previous version, first copy its full workdir from /kaggle/input here.


In [ ]:
command = [sys.executable, "scripts/run_pipeline.py",
    "--workdir", str(workdir),
    "--model", "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "--dataset", "math500", "--limit", "100", "--max-hours", "11",
    "--dtype", "float16", "--n-forced", "2", "--k-siblings", "2",
    "--sibling-stride", "4", "--sibling-max-tokens", "128",
    "--max-thinking-tokens", "2048", "--max-steps", "16"]
# GPU 0 holds the small model; this avoids spending memory on automatic sharding.
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
completed = subprocess.run(command, env=env)
print("Pipeline exit status:", completed.returncode)


In [ ]:
from IPython.display import Markdown, display
display(Markdown((workdir / "SUMMARY.md").read_text()))
print("Download:", workdir / "results.zip")
print("Retain the entire workdir, including traces/shards, for collection resume.")
